In [6]:
import jax
import jax.numpy as jnp
from jax import random
from jax.scipy.linalg import cholesky, solve_triangular
from typing import NamedTuple

class GRFParams(NamedTuple):
    f_hat: jnp.ndarray  # shape: (N_samples, n_mus)
    mus: jnp.ndarray    # shape: (N_samples, n_mus, d)
    l: jnp.ndarray      # shape: (d,)

def compute_covariance(mus, l):
    scaled = mus / l
    diff = scaled[:, None, :] - scaled[None, :, :]
    dist2 = jnp.sum(diff ** 2, axis=-1)
    return jnp.exp(-0.5 * dist2)

@jax.jit
def compute_f_hat(key, mus, l):
    cov = compute_covariance(mus, l)
    L = cholesky(cov + 1e-6 * jnp.eye(cov.shape[0]), lower=True)
    z = random.normal(key, shape=(cov.shape[0],))
    f = L @ z
    L_inv = solve_triangular(L, jnp.eye(L.shape[0]), lower=True)
    cov_inv = L_inv.T @ L_inv
    f_hat = cov_inv @ f
    return f_hat

@jax.jit
def grf_forward(f_hat, mus, l, x):
    dist2 = jnp.sum(((x[:, None, :] - mus[None, :, :]) / l) ** 2, axis=-1)
    phi = f_hat[None, :] * jnp.exp(-0.5 * dist2)
    return jnp.sum(phi, axis=-1)

def grf_batch_forward(params: GRFParams, x):
    def per_sample_forward(f_hat, mus):
        return grf_forward(f_hat, mus, params.l, x)
    return jax.vmap(per_sample_forward)(params.f_hat, params.mus)

def grf_batch_grad(params: GRFParams, x):
    def single_eval(f_hat, mus):
        func = lambda x_: grf_forward(f_hat, mus, params.l, x_)
        return jax.jacrev(func)(x)
    return jax.vmap(single_eval)(params.f_hat, params.mus)

def grf_batch_laplacian(params: GRFParams, x):
    def single_eval(f_hat, mus):
        func = lambda x_: grf_forward(f_hat, mus, params.l, x_)
        hess = jax.hessian(func)(x)
        return jnp.trace(hess, axis1=-2, axis2=-1)  # trace of Hessian = Laplacian
    return jax.vmap(single_eval)(params.f_hat, params.mus)

def generate_grfs(key, N_samples, l):
    l = jnp.array(l)
    d = l.shape[0]
    n_mus = int(jnp.ceil(1.0 / jnp.prod(l)))

    subkeys = random.split(key, N_samples * 2)
    mus = random.uniform(subkeys[:N_samples], shape=(N_samples, n_mus, d))

    def compute_sample(k, m):
        return compute_f_hat(k, m, l)

    f_hat = jax.vmap(compute_sample)(subkeys[N_samples:], mus)
    return GRFParams(f_hat=f_hat, mus=mus, l=l)

def scale_grfs(params: GRFParams, c, b):
    return GRFParams(f_hat=params.f_hat * c[:, None] + b[:, None], mus=params.mus, l=params.l)


ModuleNotFoundError: No module named 'jax'

In [4]:
!pip install jax

  Using cached jax-0.6.2-py3-none-any.whl (2.7 MB)
  Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (37.7 MB)
  Using cached ml_dtypes-0.5.1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (4.7 MB)
  Using cached jaxlib-0.6.2-cp310-cp310-manylinux2014_x86_64.whl (89.9 MB)
  Attempting uninstall: scipy
    Found existing installation: scipy 1.10.0
    Uninstalling scipy-1.10.0:
      Successfully uninstalled scipy-1.10.0
ERROR: Could not install packages due to an OSError: [Errno 16] Device or resource busy: '.nfs0000002b0285e74700001140'

